In [1]:
rm(list = ls())

# check and install CRAN packages
cran_packages <- c("agricolae", "ggplot2", "dplyr", "tibble")

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

lapply(cran_packages, library, character.only = TRUE)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




[[1]]
[1] "agricolae" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[7] "datasets"  "methods"   "base"     

[[2]]
 [1] "ggplot2"   "agricolae" "repr"      "stats"     "graphics"  "grDevices"
 [7] "utils"     "datasets"  "methods"   "base"     

[[3]]
 [1] "dplyr"     "ggplot2"   "agricolae" "repr"      "stats"     "graphics" 
 [7] "grDevices" "utils"     "datasets"  "methods"   "base"     

[[4]]
 [1] "tibble"    "dplyr"     "ggplot2"   "agricolae" "repr"      "stats"    
 [7] "graphics"  "grDevices" "utils"     "datasets"  "methods"   "base"

In [2]:
# load 1001 genome snp data
snp_data <- openxlsx::read.xlsx( "./lib/1001genomes_snp-short-indel_only_ACGTN_v3.1.vcf.genes.xlsx", startRow = 2)

snp_data <- snp_data %>%
  mutate(
    PHI = rowSums(across(c(46, 65)), na.rm = TRUE),
    PHI_density = as.numeric(PHI) / as.numeric(.data[["Length.(EXON)"]])
  )

In [3]:
prot_temp <- read.delim("./lib/all_gene_nuc_div.txt", row.names = 1, header = TRUE, sep = "\t")

sub_all_snp <- snp_data %>%
  filter(ID %in% rownames(prot_temp)) %>%
  mutate(Class = "all-protein") %>%
  filter(!is.na(ID))

In [4]:
set.seed(123)

n_sample <- 500
n_iter <- 1000

ran_set <- replicate(
  n_iter,
  sample(sub_all_snp$PHI_density, n_sample, replace = FALSE)
)

ran_median <- apply(ran_set, 1, function(x) median(x, na.rm = TRUE))

sub_ran_snp <- tibble(
  ID = paste0("random_", seq_len(n_sample)),
  PHI_density = as.numeric(ran_median),
  Class = "random"
)

In [5]:
ppr_class <- read.delim2("./lib/ppr_class.txt", sep = "\t", header = TRUE)
nlr_class <- read.delim2("./lib/nlr_class.txt", sep = "\t", header = TRUE)
gene_class <- rbind(ppr_class, nlr_class)

sub_snp <- snp_data %>%
  semi_join(gene_class, by = "ID") %>%
  left_join(gene_class %>% select(ID, Class), by = "ID") %>%
  select(ID, PHI_density, Class)

pdata <- bind_rows(sub_snp, sub_ran_snp) %>%
  mutate(
    PHI_density = as.numeric(PHI_density)
  )

level <- c(
  "HS-siRNA-PPR",
  "non-HS-siRNA-PPR",
  "non-siRNA-PPR",
  "siRNA-NLR",
  "non-siRNA-NLR",
  "random"
)

pdata <- pdata %>%
  mutate(Class = factor(Class, levels = level))

In [6]:
test <- agricolae::kruskal(
  pdata$PHI_density,
  pdata$Class,
  group = TRUE,
  p.adj = "bonferroni",
  alpha = 0.05
)

t_comp <- test$groups %>%
  as.data.frame() %>%
  rownames_to_column("group") %>%
  as_tibble() %>%
  mutate(group = factor(group, levels = level)) %>%
  arrange(group)

t_comp

group,pdata$PHI_density,groups
<fct>,<dbl>,<chr>
HS-siRNA-PPR,790.2500,a
non-HS-siRNA-PPR,775.1250,ab
non-siRNA-PPR,442.5919,b
siRNA-NLR,1067.5000,a
non-siRNA-NLR,957.2593,a
random,551.4730,b


In [7]:

p <- ggplot(pdata, aes(x = Class, y = PHI_density)) +
  geom_boxplot(aes(fill = Class), outlier.size = -1, width = 0.3) +
  geom_jitter(color = "black", size = 0.5,
              position = position_jitter(width = 0.2), alpha = 0.3) +
  scale_fill_manual(values = c(
      "HS-siRNA-PPR" = "#9E4231",
      "non-HS-siRNA-PPR" = "#30669B",
      "non-siRNA-PPR" = "#C6833F",
      "siRNA-NLR" = "#008073",
      "non-siRNA-NLR" = "#7C7CB1",
      "random" = "#787C7E"
  )) +
  labs(x = NULL, y = "PHI density") +
  coord_cartesian(ylim = c(0, 0.025)) +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_rect(fill = NA, colour = "black", linewidth = 1),
    panel.background = element_blank(),
    axis.text = element_text(colour = "black"),
    axis.text.x = element_text(colour = "black", angle = 45, hjust = 1, vjust = 1),
    axis.ticks = element_line(colour = "black"),
    legend.position = "none"
  )

ggsave(p, filename = "high_impact_variation_density.pdf", width = 4.5, height = 3)

In [8]:
df <- openxlsx::read.xlsx("./lib/homolog_srna_overlap.xlsx", sheet = 1)

df <- df %>%
  mutate(
    TPM = Count / Total * 1e6,
    Locus = factor(Locus),
    Class = factor(Class, levels = c("Specific", "Overlap"))
  ) %>%
  select(-Count, -Total)

In [9]:
p <- ggplot(df, aes(x = Locus, y = TPM, fill = Class)) +
  geom_col(color = "black", width = 0.5) +
  scale_fill_manual(values = c("Specific" = "#B5261D", "Overlap" = "#435F60")) +
  labs(x = NULL, y = "TPM of siRNAs") +
  theme(
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border = element_rect(fill = NA, colour = "black", linewidth = 1),
    panel.background = element_blank(),
    axis.text = element_text(colour = "black"),
    axis.text.x = element_text(colour = "black", angle = 45, hjust = 1, vjust = 1),
    axis.ticks = element_line(colour = "black"),
    legend.key.size = unit(0.5, "cm"),
    legend.key.height = unit(0.4, "cm"),
    legend.key.width = unit(0.4, "cm"),
    legend.title = element_text(size = 10, face = "plain"),
    legend.text = element_text(size = 10, face = "plain")
  )

ggsave(p, filename = "AT1G62670_homolog_siRNA_overlap.pdf", height = 3, width = 2.5)